In [0]:
from pyspark.sql.functions import col, when
from delta.tables import DeltaTable

In [0]:
ENV = dbutils.widgets.get("env") if dbutils.widgets.get("env") else "dev"

In [0]:
def run_silver_transformation(catalog="dev"):
    #read from bronze
    df = spark.read.table(f"{catalog}.bronze.yellow_taxi_raw")
    
    # deduplicate
    df = df.dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"])

    # creating a new column 'is_valid' 
    df = df.withColumn("is_valid", 
        when(
            (col("passenger_count").isNull()) | 
            (col("passenger_count") == 0) |
            (col("trip_distance") <= 0) | 
            (col("fare_amount") < 0) |
            (col("tpep_pickup_datetime") < "2023-01-01") |
            (col("tpep_pickup_datetime") >= "2023-02-01") |
            (col("PULocationID").isNull()) |
            (col("DOLocationID").isNull()) |
            (col("Ratecodeid").isNull()),
            False
        ).otherwise(True))
    

    # creating a new column is_valid_time
    df = df.withColumn("is_valid_time",
        when(
            col("tpep_dropoff_datetime") < col("tpep_pickup_datetime"),
            False
        ).otherwise(True))
    
    
    existing_cols = df.columns
    fill_dict = {}
    if "congestion_surcharge" in existing_cols:
        fill_dict["congestion_surcharge"] = 0
    if "airport_fee" in existing_cols:
        fill_dict["airport_fee"] = 0
    if "store_and_fwd_flag" in existing_cols:
        fill_dict["store_and_fwd_flag"] = "N"
    if fill_dict:
        print(df.columns)
        df = df.fillna(fill_dict)


    # renaming columns
    new_columns = ["vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count","trip_distance", "ratecode_id", "store_and_fwd_flag", "pu_location_id", "do_location_id", "payment_type", "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount", "congestion_surcharge", "airport_fee", "ingestion_timestamp", "source_file", "is_valid", "is_valid_time"]
    
    rename_map = {
    "VendorID": "vendor_id",
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "PULocationID": "pu_location_id",
    "DOLocationID": "do_location_id",
    "RatecodeID": "ratecode_id"
    }

    for old_name, new_name in rename_map.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)


    # creating silver table
    from delta.tables import DeltaTable

    if not spark.catalog.tableExists(f"{catalog}.silver.yellow_taxi_clean"):
        df.write.format("delta").saveAsTable(f"{catalog}.silver.yellow_taxi_clean")
    else:
        DeltaTable.forName(spark, f"{catalog}.silver.yellow_taxi_clean") \
        .alias("existing") \
        .merge(
            df.alias("new"),
            "existing.vendor_id = new.vendor_id AND existing.pickup_datetime = new.pickup_datetime AND existing.pu_location_id = new.pu_location_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

In [0]:
spark.sql("SELECT COUNT(*) FROM dev.silver.yellow_taxi_clean").show()

In [0]:
# exploration values

# counting null values
# from pyspark.sql.functions import col, sum, when
# null_counts = df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])

# checking for duplicates
# total_rows = df.count()
# unique_rows = df.dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"]).count()
# duplicate_count = total_rows - unique_rows

# checking for invalid values
# negative_fares = df.filter(col("fare_amount") <= 0).count()
# invalid_distance = df.filter(col("trip_distance") <= 0).count()